# Comparing celltypes at different conditions using pyDESEQ2

In [1]:
import scanpy as sc
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
import pandas as pd
import numpy as np
import seaborn as sns
import gseapy as gp
from gseapy import gseaplot

adata = sc.read_h5ad(r"D:\Dom\Psoriasis project\4th year data\Second Round Data\Xenium outputs - second round\Resegmented Xenium Outputs\ResolVI_subclusters.h5ad")
adata

bdata = adata[adata.obs['compartment'].str.contains('orsal')].copy()

## Cells of the same types at different timepoints

In [28]:
import decoupler as dc

celltype_bulk = {}
for celltype in ['myeloid', 'M2-like_macrophage', 'activated_basal_KC']:
    bdata_cell = bdata[bdata.obs['celltype_assignments'] == celltype ].copy()

    #filtering genes expressed by fewer than 1% cells
    min_cells = int(0.01 * bdata_cell.n_obs)
    sc.pp.filter_genes(bdata_cell, min_cells=min_cells)

    big_bulk = dc.pp.pseudobulk(
        bdata_cell,
        sample_col = 'batch_key', #defines replicates
        groups_col = None, #ignore if using batch_key, add in if using celltypes
        layer = None,
        mode = 'sum',
        empty = True
    )
    celltype_bulk[celltype] = big_bulk

In [29]:
celltype_bulk_split = {}

def isolate_sex(adata, sex_str):
    bdata = adata[adata.obs.replicate.str.contains(sex_str)].copy()
    return bdata

for name, obj in celltype_bulk.items():
    for sex in ['F', 'M']:
        celltype_bulk_split[f'{name}_{sex}'] = isolate_sex(obj, sex)


In [25]:
from pathlib import Path
results_path = Path(r"C:\Users\dbuxton\OneDrive\Desktop\biochem\Year 4\Thesis\results figures or slides\DESEQ_celltype_results")

In [26]:
#defining helper function for volcano plotting
import matplotlib.pyplot as plt
import numpy as np
from adjustText import adjust_text   # pip install adjustText

def volcano_plot(df, title, save_dir):
    df = df.dropna(subset=['log2FoldChange', 'padj'])
    df['-log10padj'] = -np.log10(df['padj'])

    fc_thresh = 1.0
    padj_thresh = 0.05
    n_labels = 20  # number of top genes to label

    fig, ax = plt.subplots(figsize=(8, 6))

    # Plot all points
    colours = np.where(
        (df['padj'] < padj_thresh) & (df['log2FoldChange'] > fc_thresh), 'firebrick',
        np.where(
            (df['padj'] < padj_thresh) & (df['log2FoldChange'] < -fc_thresh), 'steelblue', 'lightgrey'
        )
    )
    ax.scatter(df['log2FoldChange'], df['-log10padj'], c=colours, alpha=0.6, s=15)

    # Label top genes by significance
    top_genes = df.nsmallest(n_labels, 'padj')
    texts = [
        ax.text(row['log2FoldChange'], row['-log10padj'], gene, fontsize=7)
        for gene, row in top_genes.iterrows()
    ]
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.5))

    ax.axhline(-np.log10(padj_thresh), color='black', linestyle='--', linewidth=0.8)
    ax.axvline(fc_thresh, color='black', linestyle='--', linewidth=0.8)
    ax.axvline(-fc_thresh, color='black', linestyle='--', linewidth=0.8)

    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10 adjusted p-value')
    ax.set_title(title)
    
    save_path = save_dir / (title)
    plt.savefig(save_path, bbox_inches='tight', dpi = 400)
    plt.close()


In [27]:
#helper function for enrichment map plotting
from gseapy import enrichment_map
import networkx as nx
import textwrap

def enrichment_plot(pre_res, output_path):

    nodes, edges = enrichment_map(pre_res.res2d)
    G = nx.from_pandas_edgelist(edges,
                            source='src_idx',
                            target='targ_idx',
                            edge_attr=['jaccard_coef', 'overlap_coef', 'overlap_genes'])
    
    fig, ax = plt.subplots(figsize=(8, 8))
    pos = nx.layout.spiral_layout(G)

    nodelist = list(G.nodes())
    nodes_aligned = nodes.loc[nodelist]

    nx.draw_networkx_nodes(G,
                        pos=pos,
                        nodelist=nodelist,
                        cmap=plt.cm.RdYlBu,
                        node_color=list(nodes_aligned.NES),
                        node_size=list(nodes_aligned.Hits_ratio * 1000))

    # wrap labels and draw with clip_on=False
    wrapped_labels = {
        node: '\n'.join(textwrap.wrap(term, width=20))
        for node, term in nodes_aligned.Term.to_dict().items()
    }
    label_objs = nx.draw_networkx_labels(G, pos=pos, labels=wrapped_labels, font_size=7)
    for _, text in label_objs.items():
        text.set_clip_on(False)

    edge_weight = nx.get_edge_attributes(G, 'jaccard_coef').values()
    nx.draw_networkx_edges(G,
                        pos=pos,
                        width=list(map(lambda x: x*10, edge_weight)),
                        edge_color='#CDDBD4')
    
    x_values = [pos[n][0] for n in G.nodes()]
    y_values = [pos[n][1] for n in G.nodes()]

    max_label_len = max(len(t) for t in nodes_aligned.Term)
    pad = 0.3 + (max_label_len * 0.01)

    ax.set_xlim(min(x_values) - pad, max(x_values) + pad)
    ax.set_ylim(min(y_values) - pad, max(y_values) + pad)

    plt.tight_layout()
    plt.savefig(output_path, dpi=400, bbox_inches='tight')
    plt.close()

In [30]:
#run on a whole dictionary - single object workflow in following cells
failed_deseq2 = []
low_DEs = []
small_GSEAs = []
failed_enrichments = []

tested_cond = "D10IMQ"
reference_cond = "D7IMQ"

for name, bulk in celltype_bulk_split.items():
    #make a deseq data set from your anndata
    
    ###avoid recalculating if already done###
    if Path(results_path/ f'{name}_{reference_cond}_{tested_cond}_DEgenes.csv').exists(): #bit jammy, but if D3 is calced, then so are D7/D10
        continue

    dds = DeseqDataSet(counts = bulk.X,
                   metadata = bulk.obs,
                   design_factors = 'condition')
    dds.var = bulk.var


    #run deseq2 on it
    try:
        dds.deseq2()
    except:
        failed_deseq2.append(f'{name}')
        continue

    #get the stats
    for day in ['D10IMQ']:
        print(name.split("_")[0])
        print("="*80)


        stat_res = DeseqStats(dds, contrast = ['condition', day, reference_cond]) #pairwise comparison of days with each condition
        stat_res.summary()
        #get diffexp dataframe
        res = stat_res.results_df
        res_file = f'{name}_{reference_cond}_{day}_DEgenes.csv'
        res.to_csv(results_path / res_file) #save table with stats of DE genes
        
        #filtering any genes with baseMean <10, choice is somewhat arbitrary
        res = res[res.baseMean >10]
        #then filter all those that are not significant or large enough
        sigs = res[(abs(res.log2FoldChange) > 0.58)&(res.padj <0.05)] #again, log2fc threshold is bit arbitrary
        #check if yap1 is in the list of significantly expressed genes
        if 'Yap1' in sigs.index.to_list():
            print(f'{name}_{reference_cond}_{day} has a significant change in YAP1')
        
        #plot VOLCANO PLOT and save
        volcano_plot_name = f'{name}_{reference_cond}_{day}_volcano.png'

        save_dir = results_path / 'Volcano_plots'
        volcano_plot(res, volcano_plot_name, save_dir)
        
        #plot CLUSTERMAP for significant genes
        cluster_map_name = f'{name}_{reference_cond}_{day}_cluster.png'
        cluster_map_path = results_path / 'clustermaps' / cluster_map_name
        if len(sigs) >2:
            dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])
            dds_sigs = dds[:, sigs.index].copy()

            grapher = pd.DataFrame(dds_sigs.layers['log1p'].T,
                        index = dds_sigs.var_names, columns = dds_sigs.obs_names)
            
            clustermap = sns.clustermap(grapher, z_score = 0, cmap = 'RdYlBu_r')
            clustermap.savefig(cluster_map_path, bbox_inches = 'tight', dpi = 400)
            plt.close(clustermap.fig)
        else:
            low_DEs.append(f'{name}_{reference_cond}_{day}')

        #do GSEA on your diffexped genes
        ranking = res['stat'].dropna().sort_values(ascending = False)

        for gene_set in ['GO_Biological_Process_2026', 'KEGG_2026']:
            if gene_set == 'GO_Biological_Process_2026':
                database = 'GO'
            else:
                database = 'KEGG' 
            
            try:
                pre_res = gp.prerank(rnk = ranking,
                        gene_sets = gene_set, #this can be a list, e.g. if you wanted to look at molecular function, or KEGG
                        organism = 'mouse',
                        seed = 6, permutation_num=100)
                
                print(name.split("_")[1])
                if name.split("_")[1] == 'F':
                    sex = 'female'
                else:
                    sex = 'male'
                
                dot_title = f'{name.split("_")[0]} {sex} {reference_cond} {day} {database}'
                ax = gp.dotplot(pre_res.res2d,
                            column="FDR q-val",
                            title=dot_title,
                            cmap=plt.cm.RdBu_r,
                            size=4, # adjust dot size
                            figsize=(4,5), 
                            cutoff=0.25, 
                            show_ring=False,
                            ofname=str(results_path / 'GSEA_plots' / f'{name.split("_")[0]} {sex} {reference_cond} {day} {database}_dotplot.png'))

                out = []
                for term in list(pre_res.results):
                    out.append([term,
                                pre_res.results[term]['fdr'],
                                pre_res.results[term]['es'],
                                pre_res.results[term]['nes']])
                print("succeeded 2")
                    
                out_df = pd.DataFrame(out, columns = ['Term', 'fdr', 'es', 'nes']).sort_values('fdr').reset_index(drop = True)
                out_df.sort_values('nes', ascending = False)
                print("succeeded 3")



                out_df.to_csv(results_path / 'GSEA_csvs' / f'{name}_{reference_cond}_{day}_{database}.csv')

                #plotting the ENRICHMENT MAP
                output_path = results_path / 'GSEA_plots'/ f'{name}_{reference_cond}_{day}_{database}.png'
                try:
                    enrichment_plot(pre_res, output_path)
                except:
                    print(f'no signficant {database} terms for {name}_D3IMQ{day}')
                    failed_enrichments.append(f'{name}_{reference_cond}_{day}_{database}')
            except:
                small_GSEAs.append(f'{name}_{reference_cond}_{day}_{database}')


print(failed_deseq2) 
print(low_DEs) 
print(small_GSEAs) 
print(failed_enrichments)

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_36868\1071917369.py:17: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.



Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 0.18 seconds.

Fitting dispersion trend curve...
... done in 0.06 seconds.

Fitting MAP dispersions...
... done in 0.17 seconds.

Fitting LFCs...
... done in 0.21 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 0 outlier genes.

Running Wald tests...


myeloid


... done in 0.21 seconds.



...

Log2 fold change & Wald test p-value: condition D10IMQ vs D7IMQ
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
Aatf      11.911078        0.294637  0.341063  0.863877  0.387655  0.882053
Abca1      3.199020        0.466412  0.499797  0.933202  0.350716       NaN
Abca7     14.366296        0.147973  0.305281  0.484712  0.627881  0.939325
Abcb1b     1.992907        0.627134  0.527424  1.189052  0.234419       NaN
Abcb8      1.878819       -0.644640  0.597801 -1.078351  0.280877       NaN
...             ...             ...       ...       ...       ...       ...
Zmpste24   3.313789       -0.193975  0.490149 -0.395747  0.692292       NaN
Zmynd19    9.175954        0.129121  0.538782  0.239653  0.810599  0.971998
Zscan21    1.160510       -0.553932  0.550546 -1.006150  0.314344       NaN
Zyx        0.954079       -1.233214  0.923973 -1.334686  0.181979       NaN
Zzef1      7.170542        0.046109  0.388016  0.118832  0.905408  0.990865

[2276 rows x 6 columns]

C:\Users\dbuxton\AppData\Local\Temp\ipykernel_36868\1071917369.py:17: DeprecationWarning: design_factors is deprecated and will soon be removed.Please consider providing a formulaic formula using the design argumentinstead.
  dds = DeseqDataSet(counts = bulk.X,
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.30 seconds.

Fitting dispersion trend curve...
d:\Dom\Virtual_Environments\napari_registration_project\.venv\Lib\site-packages\pydeseq2\dds.py:822: UserWarning: The dispersion trend curve fitting did not converge. Switching to a mean-based dispersion trend.
  self._fit_parametric_dispersion_trend(vst)
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.42 seconds.

Fitting LFCs...
... done in 0.35 seconds.

Calculating cook's distance...
... done in 0.01 seconds.

Replacing 0 outlier genes.

Running Wald tests...


activated


... done in 0.30 seconds.



Log2 fold change & Wald test p-value: condition D10IMQ vs D7IMQ
           baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
Aatf       6.192429        0.041528  0.311071  0.133500  0.893798  0.971244
Abca1     10.612739        0.075828  0.381444  0.198791  0.842426  0.966692
Abca7      5.153575        0.392732  0.269048  1.459707  0.144371  0.477779
Abcb1b     1.068671       -0.060328  0.664691 -0.090760  0.927683       NaN
Abcb8      0.909163       -1.240398  0.726549 -1.707245  0.087776       NaN
...             ...             ...       ...       ...       ...       ...
Zmpste24   2.654511        0.145566  0.369922  0.393504  0.693948       NaN
Zmynd19    1.479072        0.162690  0.558701  0.291194  0.770903       NaN
Zscan21    0.856156        0.794662  0.584594  1.359338  0.174039       NaN
Zyx        0.998013       -2.287711  1.045389 -2.188383  0.028642       NaN
Zzef1      2.657603        0.108436  0.408372  0.265533  0.790599       NaN

[2276 rows x 6 columns]

In [22]:
failed_deseq2 = []
low_DEs = []
small_GSEAs = []
failed_enrichments = []